# py123d → FiftyOne: Multi-Dataset Viewer with CLIP Embeddings

Downloads ~900 MB of data from two fully public autonomous driving datasets — **Argoverse 2** (AWS S3, no account) and **PandaSet** (HuggingFace, free account required) — converts them via py123d's unified Arrow format, loads both into a single FiftyOne dataset tagged by source, and computes CLIP embeddings per camera direction for cross-dataset exploration.

| Dataset | Logs | Iterations | Approx. size |
|---------|------|------------|--------------|
| Argoverse 2 | 3 | all (157 each) | ~750 MB |
| PandaSet | 3 | all (80 each) | ~150 MB |
| **Total** | | **~700+ camera samples per direction** | **~900 MB** |

---

## ⚠️ Setup Notes, Tricks and Tips

- If pyenv intercepts `pip`/`python`, use the venv binaries directly: `~/py123d-fo-env/bin/pip`
- The concrete py123d API class is `ArrowSceneAPI` — the base `SceneAPI` is abstract and cannot be instantiated
- There is no `iter_frames()` method — iteration is index-based via `get_*_at_iteration(i, id)`
- Box labels must be cast with `str().split('.')[-1]` — they are enums e.g. `AV2SensorBoxDetectionLabel.BICYCLIST`
- `np.ptp()` must be used instead of `arr.ptp()` (removed in NumPy 2.0)
- Use `matplotlib.colormaps["jet"]` instead of `plt.get_cmap("jet")` (deprecated in Matplotlib 3.7)
- PandaSet anonymous HuggingFace downloads are throttled — a free HF account and token are required
- Box coordinates are in world frame and must be transformed to ego frame to align with the point cloud
- `numpy<2` must be installed before torch — if torch is already installed, force-reinstall numpy: `pip install 'numpy<2' --force-reinstall` then restart Jupyter
- t-SNE `perplexity` must be less than the number of samples
- t-SNE `pca_dims` must be less than or equal to the number of samples
- After overwriting a dataset, relaunch the app — the old session points to a deleted dataset
- Delete existing brain runs before re-running embedding cells to avoid brain_key conflicts
- If the dataset was already built in a previous session, use `fo.load_dataset('py123d_combined')` instead of rebuilding

## Step 1: Create the virtual environment and place this notebook inside it

Run all of these in your **terminal**. The notebook is moved into the venv directory so Jupyter always finds it.

```bash
python -m venv ~/py123d-fo-env
source ~/py123d-fo-env/bin/activate

# Use venv binaries directly to avoid pyenv shim interference
~/py123d-fo-env/bin/pip install --upgrade pip

# numpy<2 must come before torch to avoid NumPy 2.x incompatibility
# If torch is already installed, force-reinstall numpy and restart Jupyter:
# ~/py123d-fo-env/bin/pip install 'numpy<2' --force-reinstall
~/py123d-fo-env/bin/pip install "numpy<2"
~/py123d-fo-env/bin/pip install "py123d[av2,pandaset,hf]" fiftyone open3d opencv-python matplotlib jupyter ipykernel
~/py123d-fo-env/bin/pip install torch torchvision

# Register as a Jupyter kernel
~/py123d-fo-env/bin/python -m ipykernel install --user --name py123d-fo-env --display-name "py123d-fo-env"

# Move this notebook into the venv directory
mv ~/Downloads/py123d_to_fiftyone.ipynb ~/py123d-fo-env/

# Launch Jupyter
~/py123d-fo-env/bin/jupyter notebook ~/py123d-fo-env/py123d_to_fiftyone.ipynb
```

When Jupyter opens, select **Kernel → Change Kernel → py123d-fo-env**.

## Step 2: Install dependencies into the venv

Uses `sys.executable` to guarantee packages land in the active venv. Safe to re-run — pip skips already-installed packages.

After running this cell for the first time, **Kernel → Restart Kernel** then continue from Step 3.

In [ ]:
import sys
import subprocess

def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)

# numpy<2 must be installed before torch
pip_install("numpy<2")
pip_install("py123d[av2,pandaset,hf]", "fiftyone", "open3d", "opencv-python", "matplotlib")
pip_install("torch", "torchvision")

print("All packages installed — restart the kernel now (Kernel → Restart Kernel) then continue from Step 3")

## Step 3: Imports and environment check

Verify numpy is `<2.0` before continuing. If it shows `2.x`, stop and run in terminal:
```bash
~/py123d-fo-env/bin/pip install 'numpy<2' --force-reinstall
```
Then restart Jupyter completely (not just the kernel).

In [ ]:
import os
import sys

# Suppress OpenMP duplicate library warning (harmless on macOS)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print(sys.executable)  # must contain py123d-fo-env, not .pyenv

import py123d
print("py123d:", py123d.__file__)  # must contain py123d-fo-env

import numpy as np
print("numpy:", np.__version__)  # must be <2.0 — if not, see note above
assert tuple(int(x) for x in np.__version__.split(".")[:2]) < (2, 0), \
    "numpy>=2 detected — run: pip install 'numpy<2' --force-reinstall, then restart Jupyter"

import fiftyone as fo
import fiftyone.brain as fob
import open3d as o3d
import cv2
import matplotlib
import huggingface_hub
import torch
print("torch:", torch.__version__)
print("All imports OK")

## Step 4: Log in to HuggingFace

Required to avoid anonymous download throttling for PandaSet.

1. Sign up at [huggingface.co](https://huggingface.co)
2. Generate a **read** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
3. Paste it when prompted below

Skip this step if you already ran it and are resuming a previous session.

In [ ]:
from huggingface_hub import login, whoami

login()  # token is not stored in the notebook
user = whoami()
print(f"Logged in as: {user['name']}")

## Step 5: Set up the data directory

In [ ]:
data_root = os.path.expanduser("~/py123d_data")
os.makedirs(data_root, exist_ok=True)
os.environ["PY123D_DATA_ROOT"] = data_root

print(f"PY123D_DATA_ROOT = {data_root}")
print(f"Writable: {os.access(data_root, os.W_OK)}")

## Step 6: Download datasets

**Skip these download cells if you already ran them** — the converted data persists on disk in `~/py123d_data`. Jump straight to the confirmation cells below to set the log ID variables.

### 6a: Argoverse 2 — 3 logs (~750 MB)

3 logs × 157 iterations = 471 front camera frames. No account needed.

In [ ]:
# SKIP IF ALREADY DOWNLOADED — data persists on disk
# No inline # comments — causes zsh parsing errors if run in terminal
!py123d-conversion \
    dataset=av2-sensor-stream \
    dataset.parser.splits='[av2-sensor_val]' \
    dataset.parser.downloader.num_logs=3

In [ ]:
# Always run this cell to set av2_log_ids
av2_log_dir = os.path.join(data_root, "logs", "av2-sensor_val")
av2_log_ids = os.listdir(av2_log_dir)
print(f"AV2 logs: {av2_log_ids}")
assert len(av2_log_ids) > 0, "No AV2 logs found — run the download cell above"

### 6b: PandaSet — 3 logs via HuggingFace mirror (~150 MB)

In [ ]:
# SKIP IF ALREADY DOWNLOADED — data persists on disk
!py123d-conversion \
    dataset=pandaset-stream \
    dataset.parser.downloader.num_logs=3

In [ ]:
# Always run this cell to set panda_log_ids
panda_base = os.path.join(data_root, "logs")
panda_splits = [d for d in os.listdir(panda_base) if "pandaset" in d.lower()]
print(f"PandaSet split dirs: {panda_splits}")

panda_log_dir = os.path.join(panda_base, panda_splits[0])
panda_log_ids = os.listdir(panda_log_dir)
print(f"PandaSet logs: {panda_log_ids}")
assert len(panda_log_ids) > 0, "No PandaSet logs found — run the download cell above"

In [ ]:
!du -sh {data_root}

## Step 7: Helper functions

In [ ]:
JET = matplotlib.colormaps["jet"]  # colormaps[] not get_cmap() — deprecated in Matplotlib 3.7

def colorize_by_height(xyz):
    """Color LiDAR points by Z height: blue=low, red=high."""
    z = xyz[:, 2]
    z_norm = (z - z.min()) / (np.ptp(z) + 1e-6)  # np.ptp() — arr.ptp() removed in NumPy 2.0
    return JET(z_norm)[:, :3]

def save_image(image_array, path):
    """Save a numpy RGB image to disk as JPEG."""
    cv2.imwrite(path, cv2.cvtColor(image_array, cv2.COLOR_RGB2BGR))

def save_pcd(xyz, path):
    """Save a height-colored point cloud to disk as PCD."""
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(xyz)
    pcd.colors = o3d.utility.Vector3dVector(colorize_by_height(xyz))
    o3d.io.write_point_cloud(path, pcd)

def make_3d_detections_ego(boxes, ego):
    """
    Transform box centers from world frame into ego (vehicle) frame.
    - ego.center_se3.inverse is a property, not a method — no () needed
    - label is an enum — split on . to get clean name
    """
    ego_inv_mat = ego.center_se3.inverse.transformation_matrix
    detections = []
    for box in boxes:
        world_xyz = np.array([box.center_se3.x, box.center_se3.y, box.center_se3.z, 1.0])
        ego_xyz = ego_inv_mat @ world_xyz
        rel_yaw = box.center_se3.yaw - ego.center_se3.yaw
        detections.append(fo.Detection(
            label=str(box.attributes.label).split(".")[-1],
            location=[float(ego_xyz[0]), float(ego_xyz[1]), float(ego_xyz[2])],
            dimensions=[box.bounding_box_se3.length,
                        box.bounding_box_se3.width,
                        box.bounding_box_se3.height],
            rotation=[box.center_se3.roll,
                      box.center_se3.pitch,
                      float(rel_yaw)],
        ))
    return fo.Detections(detections=detections)

print("Helpers defined")

## Step 8: Load samples from a log

Camera names are normalised to shared directional labels (`front`, `side_left`, `rear`, etc.) so AV2 and PandaSet land in the same FiftyOne slices — this enables per-direction cross-dataset embeddings.

In [ ]:
from py123d.api.scene.arrow.arrow_scene_api import ArrowSceneAPI

PCD_TEMP = "/tmp/py123d_fo_pcd"
IMG_TEMP = "/tmp/py123d_fo_img"
os.makedirs(PCD_TEMP, exist_ok=True)
os.makedirs(IMG_TEMP, exist_ok=True)

# Map dataset-specific camera names to shared directional labels
AV2_CAM_MAP = {
    "ring_front_center":  "front",
    "ring_front_left":    "front_left",
    "ring_front_right":   "front_right",
    "ring_side_left":     "side_left",
    "ring_side_right":    "side_right",
    "ring_rear_left":     "rear_left",
    "ring_rear_right":    "rear_right",
    "stereo_front_left":  "stereo_front_left",
    "stereo_front_right": "stereo_front_right",
}

PANDASET_CAM_MAP = {
    "front_camera":       "front",
    "front_left_camera":  "front_left",
    "front_right_camera": "front_right",
    "left_camera":        "side_left",
    "right_camera":       "side_right",
    "back_camera":        "rear",
}

CAM_MAPS = {
    "av2":      AV2_CAM_MAP,
    "pandaset": PANDASET_CAM_MAP,
}

def load_log_samples(log_dir, log_id, source_tag, max_iterations=None):
    """
    Load frames from a py123d log into FiftyOne samples.
    Camera names are normalised to shared directional labels.
    Boxes are transformed from world to ego frame.
    """
    scene = ArrowSceneAPI(os.path.join(log_dir, log_id))
    n = scene.number_of_iterations if max_iterations is None else min(max_iterations, scene.number_of_iterations)
    cam_map = CAM_MAPS.get(source_tag, {})

    print(f"  {source_tag} / {log_id}: loading {n}/{scene.number_of_iterations} iterations")
    print(f"  Cameras: {scene.available_camera_names}")
    print(f"  LiDAR:   {scene.available_lidar_names}")

    lidar_id = scene.available_lidar_ids[
        scene.available_lidar_names.index("lidar_merged")
        if "lidar_merged" in scene.available_lidar_names
        else 0
    ]

    samples = []
    for i in range(n):
        group = fo.Group()

        ego = scene.get_ego_state_se3_at_iteration(i)
        box_data = scene.get_box_detections_se3_at_iteration(i)
        boxes = box_data.box_detections or []

        # Camera slices — normalised to shared directional names
        for cam_id, cam_name in zip(scene.available_camera_ids, scene.available_camera_names):
            slice_name = cam_map.get(cam_name, cam_name)
            cam = scene.get_camera_at_iteration(i, cam_id)
            img_path = os.path.join(IMG_TEMP, f"{source_tag}_{log_id}_{cam_name}_{i:04d}.jpg")
            save_image(cam.image, img_path)
            samples.append(fo.Sample(
                filepath=img_path,
                group=group.element(slice_name),
                source=source_tag,
                log_id=log_id,
                camera_direction=slice_name,
            ))

        # LiDAR slice with ego-relative 3D bounding boxes
        lidar = scene.get_lidar_at_iteration(i, lidar_id)
        pcd_path = os.path.join(PCD_TEMP, f"{source_tag}_{log_id}_{i:04d}.pcd")
        save_pcd(lidar.xyz, pcd_path)

        samples.append(fo.Sample(
            filepath=pcd_path,
            group=group.element("lidar"),
            ground_truth=make_3d_detections_ego(boxes, ego),
            source=source_tag,
            log_id=log_id,
        ))

    print(f"  → {len(samples)} samples built")
    return samples

print("load_log_samples defined")

## Step 9: Load or build the dataset

If the dataset was already built in a previous session, load it directly — no need to rebuild. Only run the build cell if this is the first time or you want to start fresh.

In [ ]:
# Load existing dataset if available, otherwise build from scratch
DATASET_NAME = "py123d_combined"

if DATASET_NAME in fo.list_datasets():
    dataset = fo.load_dataset(DATASET_NAME)
    print(f"Loaded existing dataset: {len(dataset)} samples")
    print(f"Group slices: {dataset.group_media_types}")
else:
    print("Dataset not found — building from scratch...")

    dataset = fo.Dataset(DATASET_NAME, overwrite=True)
    dataset.add_group_field("group", default="front")

    all_samples = []

    print("\nLoading AV2...")
    for log_id in av2_log_ids:
        all_samples += load_log_samples(av2_log_dir, log_id, source_tag="av2")

    print("\nLoading PandaSet...")
    for log_id in panda_log_ids:
        all_samples += load_log_samples(panda_log_dir, log_id, source_tag="pandaset")

    dataset.add_samples(all_samples)
    dataset.persistent = True  # persist across sessions
    dataset.save()
    print(f"\nDataset built: {len(dataset)} samples")
    print(f"Group slices: {dataset.group_media_types}")

## Step 10: Launch FiftyOne

**Navigation tips:**
- Switch camera/LiDAR slices in the **Group** panel on the right
- Press **⌘+1** for top-down LiDAR view, **⌘+3** for bottom view
- Toggle `ground_truth` in the Labels panel to show/hide 3D bounding boxes
- LiDAR points are colored by height: **blue = low, red = high**
- To see ground truth labels switch the group slice to **lidar**
- Filter by `source` to view AV2 or PandaSet separately
- Filter by `camera_direction` to view a specific camera angle

In [ ]:
session = fo.launch_app(dataset)
print(f"FiftyOne running at: {session.url}")

## Step 11: Compute CLIP embeddings per camera direction

Embeddings are computed separately for front, side, and rear cameras. This reveals:
- Whether AV2 and PandaSet cluster separately per direction (domain gap)
- Whether front vs rear vs side cameras occupy different visual distributions
- Outlier frames and rare scenarios within each direction

Notes:
- `select_group_slices()` is required — grouped datasets not supported directly
- `perplexity` and `pca_dims` are set automatically based on sample count
- Each direction gets its own `brain_key` so all are available simultaneously in the app
- Existing brain runs are deleted before recomputing to avoid key conflicts

In [ ]:
def compute_direction_embeddings(dataset, slice_name, brain_key):
    """
    Compute CLIP + t-SNE embeddings for a single camera direction slice.
    - Deletes existing brain run to avoid re-run conflicts
    - Skips if fewer than 10 samples
    - Automatically sets perplexity and pca_dims from sample count
    """
    # Delete existing brain run to avoid brain_key conflict on re-run
    if brain_key in dataset.list_brain_runs():
        dataset.delete_brain_run(brain_key)
        print(f"  Deleted existing brain run: {brain_key}")

    view = dataset.select_group_slices(slice_name)
    n = len(view)
    print(f"\n--- {slice_name} ({n} samples) ---")

    if n < 10:
        print(f"  Skipping — too few samples ({n})")
        return None

    perplexity = max(5, min(50, n // 10))  # ~10% of sample count, capped at 50
    pca_dims = min(20, n)                  # cannot exceed sample count
    print(f"  perplexity={perplexity}, pca_dims={pca_dims}")

    results = fob.compute_visualization(
        view,
        model="clip-vit-base32-torch",
        brain_key=brain_key,
        method="tsne",
        num_dims=2,
        pca_dims=pca_dims,
        perplexity=perplexity,
    )
    print(f"  ✓ Done")
    return results

print("compute_direction_embeddings defined")

In [ ]:
# Front camera — largest overlap between AV2 and PandaSet, most informative for domain gap
front_results = compute_direction_embeddings(dataset, "front", "clip_front")

In [ ]:
# Side cameras — very different visual distribution from forward-facing
side_left_results  = compute_direction_embeddings(dataset, "side_left",  "clip_side_left")
side_right_results = compute_direction_embeddings(dataset, "side_right", "clip_side_right")

In [ ]:
# Rear cameras — most visually distinct from front
# AV2 has rear_left/rear_right; PandaSet has a single rear
rear_results       = compute_direction_embeddings(dataset, "rear",       "clip_rear")
rear_left_results  = compute_direction_embeddings(dataset, "rear_left",  "clip_rear_left")
rear_right_results = compute_direction_embeddings(dataset, "rear_right", "clip_rear_right")

## Step 12: Explore embeddings in the app

Set the session view to a specific camera direction, then open the **Embeddings** panel and select a brain key.

**What to look for:**
- Color by `source` → do AV2 and PandaSet cluster separately? That's a domain gap
- Color by `log_id` → does variation cluster by log or mix across them?
- Lasso a tight cluster → what scene type does it represent?
- Lasso outliers far from clusters → these are your rarest, most valuable frames

In [ ]:
# Relaunch app to pick up new brain runs
# (old session may point to deleted dataset if dataset was rebuilt)
session = fo.launch_app(dataset)

# Set view to front camera so Embeddings panel shows clip_front
session.view = dataset.select_group_slices("front")
print(f"App running at: {session.url}")
print("Open Embeddings panel → select 'clip_front' → color by 'source'")

In [ ]:
# Switch to side camera view
session.view = dataset.select_group_slices("side_left")
print("Switch brain key to 'clip_side_left' in the Embeddings panel")

In [ ]:
# Switch to rear camera view (PandaSet only — AV2 splits into rear_left/rear_right)
session.view = dataset.select_group_slices("rear")
print("Switch brain key to 'clip_rear' in the Embeddings panel")

## Step 13: Dataset statistics

In [ ]:
# Sample count per source
for source in ["av2", "pandaset"]:
    view = dataset.match(fo.ViewField("source") == source)
    print(f"{source}: {len(view)} samples")

In [ ]:
# Label distribution across both datasets (LiDAR slice only)
from collections import Counter

dataset.group_slice = "lidar"
label_counts = Counter(
    det.label
    for s in dataset
    if s.ground_truth is not None
    for det in s.ground_truth.detections
)
print("Label distribution:")
for label, count in label_counts.most_common():
    print(f"  {label}: {count}")

In [ ]:
# Reset to full combined view
# Note: must be None or a DatasetView — session.view = dataset raises ValueError
session.view = None

# session.close()